# Lesson 2: Mini-batches and validation

Upgrade the bigram model to train on random windows and measure performance on held-out text.

**How to run:** Select a Python kernel with PyTorch installed, then run each code cell from top to bottom with **Shift+Enter**. This notebook is self-contained; no other notebook needs to run first. Restart the kernel and run from the top to reset the experiment.

**Source:** This lesson was developed from the [reference conversation's roadmap](https://chatgpt.com/share/6aa56bca-4a1c-83e9-9153-1edcc7ff7e40). The reference supplies Lesson 1 and a topic outline; Lessons 2–12 are newly written implementations of those topics. Small examples demonstrate the mechanics; they are not trained assistants.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
# Small tensors can be slower with many CPU threads.
torch.set_num_threads(1)
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)


## Load and tokenize text

We reuse `input.txt`, resolving it from the notebook folder or repository root. `B` means batch size, `T` means context length, and `C` means vector width. Our file is only 80 characters, so the validation scores are noisy and text generation will be limited.


In [ ]:
input_path = Path('input.txt')
if not input_path.is_file():
    input_path = Path('chatgpt/input.txt')
text = input_path.read_text(encoding='utf-8')
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return ''.join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.8 * len(data))
train_data, val_data = data[:split], data[split:]
block_size = min(8, len(train_data) - 1, len(val_data) - 1)
if block_size < 1:
    raise ValueError('input.txt needs enough text for train and validation sequences.')
batch_size = 4
print('Characters:', vocab_size, '| train:', len(train_data), '| validation:', len(val_data))
print('Context length:', block_size)


## Draw random batches

Choose starting positions within one split. Targets are the same window shifted right by one token. No window crosses from training into validation. The tokenizer vocabulary uses the full text so every validation character has an ID; model weights are updated only on training tokens.


In [ ]:
def get_batch(split_name='train'):
    if split_name not in ('train', 'val'):
        raise ValueError("Choose 'train' or 'val'.")
    source = train_data if split_name == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch()
print('Input shape:', xb.shape, '| target shape:', yb.shape)
print('Input :', repr(decode(xb[0])))
print('Target:', repr(decode(yb[0])))
assert torch.equal(xb[:, 1:], yb[:, :-1])


## A batched bigram model

Flatten the batch and time axes for cross entropy. Although each batch contains a window, this model still uses only the current character to predict the next one.


In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.table(idx)  # [B, T, vocabulary]
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))
        return logits, loss

model = BigramLanguageModel(vocab_size).to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))


## Estimate train and validation loss

Evaluation disables gradients and temporarily switches the model to evaluation mode. A rising validation loss while training loss falls can indicate overfitting.


In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_batches=10):
    was_training = model.training
    model.eval()
    results = {}
    for name in ('train', 'val'):
        losses = []
        for _ in range(eval_batches):
            x, y = get_batch(name)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[name] = sum(losses) / len(losses)
    model.train(was_training)
    return results

print('Before training:', estimate_loss(model))


## Train a small model

This cell continues training if rerun. Rerun the model-creation cell first for a fresh model. The small default run teaches the workflow; useful generation needs substantially more text and training.


In [ ]:
training_steps = 120
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()
for step in range(training_steps):
    x, y = get_batch()
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 40 == 0 or step == training_steps - 1:
        print(f'step {step}: loss {loss.item():.4f}')


In [ ]:
print('After training:', estimate_loss(model))


## Try it yourself

Change `batch_size` to 8, then rerun from the batch cell. Compare learning rates `3e-3` and `1e-2` from fresh models. Explain why a longer window does not give a bigram model a longer memory.
